In [ ]:
import os

# SPEED FIX: Using pure-Python protobuf makes TF graph building and IO much slower.
# Removing this preserves correctness (protobuf implementation choice does not change model semantics)
# and typically saves significant wall-clock time in Kaggle environments.
# os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

import gc
import numpy as np
import pandas as pd
import cv2

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import DenseNet121

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    _CPU = os.cpu_count() or 2
    tf.config.threading.set_intra_op_parallelism_threads(min(8, _CPU))
    tf.config.threading.set_inter_op_parallelism_threads(2)
except Exception:
    pass

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

try:
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
except Exception:
    pass

try:
    cv2.setNumThreads(0)
    cv2.ocl.setUseOpenCL(False)
except Exception:
    pass




In [ ]:
"""
    Config
"""
IMG_SIZE = 224
BATCH_SIZE = 16


def crop_image_from_gray(img, tol=7):
    if img is None:
        return None

    if img.ndim == 2:
        mask = (img > tol).astype(np.uint8)
        pts = cv2.findNonZero(mask)
        if pts is None:
            return img
        x, y, w, h = cv2.boundingRect(pts)
        return img[y : y + h, x : x + w]

    if img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = (gray_img > tol).astype(np.uint8)
        pts = cv2.findNonZero(mask)
        if pts is None:
            return img
        x, y, w, h = cv2.boundingRect(pts)
        return img[y : y + h, x : x + w, :]

    return img


def load_ben_color(image, sigmaX=10):
    image = crop_image_from_gray(image)
    if image is None:
        return None
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = cv2.addWeighted(image, 4, cv2.GaussianBlur(image, (0, 0), sigmaX), -4, 128)
    return image.astype("float32") / 255.0


def preprocessing(image, sigmaX=10):
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    image = crop_image_from_gray(image).astype("uint8")
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = cv2.addWeighted(image, 4, cv2.GaussianBlur(image, (0, 0), sigmaX), -4, 128)
    return image.astype("float32") / 255.0


_TOL_INT = 7
_SIGMAX = 10.0

_GAUSS_KSIZE = 61
_k = tf.cast(tf.range(-(_GAUSS_KSIZE // 2), _GAUSS_KSIZE // 2 + 1), tf.float32)
_xx, _yy = tf.meshgrid(_k, _k)
_kernel2d = tf.exp(
    -(_xx * _xx + _yy * _yy) / (2.0 * (tf.cast(_SIGMAX, tf.float32) ** 2))
)
_kernel2d = _kernel2d / tf.reduce_sum(_kernel2d)
_GAUSS_KERNEL4D = _kernel2d[:, :, tf.newaxis, tf.newaxis]  # [K,K,1,1]
_GAUSS_KERNEL4D = tf.tile(
    _GAUSS_KERNEL4D, [1, 1, 3, 1]
)  # [K,K,3,1] depthwise per-channel


@tf.function
def _tf_crop_from_gray(img_u8, tol=_TOL_INT):
    gray = tf.image.rgb_to_grayscale(img_u8)  # uint8
    gray2 = tf.squeeze(gray, axis=-1)  # [H,W]
    mask = gray2 > tf.cast(tol, gray2.dtype)

    def _no_crop():
        return img_u8

    def _do_crop():
        ys = tf.where(tf.reduce_any(mask, axis=1))[:, 0]
        xs = tf.where(tf.reduce_any(mask, axis=0))[:, 0]
        y0 = tf.reduce_min(ys)
        y1 = tf.reduce_max(ys) + 1
        x0 = tf.reduce_min(xs)
        x1 = tf.reduce_max(xs) + 1
        return img_u8[y0:y1, x0:x1, :]

    return tf.cond(tf.reduce_any(mask), _do_crop, _no_crop)


@tf.function
def _tf_gaussian_blur(img_f32):
    img4 = img_f32[tf.newaxis, ...]  # [1,H,W,3]
    blurred = tf.nn.depthwise_conv2d(
        img4,
        tf.cast(_GAUSS_KERNEL4D, img_f32.dtype),
        strides=[1, 1, 1, 1],
        padding="SAME",
    )
    return blurred[0]


@tf.function
def _tf_load_ben_preprocess(path):
    bytes_ = tf.io.read_file(path)
    img = tf.image.decode_png(bytes_, channels=3)  # uint8 RGB
    img = _tf_crop_from_gray(img, tol=_TOL_INT)
    img = tf.image.resize(
        img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR
    )
    img_f = tf.cast(img, tf.float32)
    blur = _tf_gaussian_blur(img_f)
    img_f = 4.0 * img_f + (-4.0) * blur + 128.0
    img_f = tf.clip_by_value(img_f, 0.0, 255.0) * (1.0 / 255.0)
    img_f.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img_f




In [ ]:
EFF_MODEL_PATH = "../input/eff-b0-model/eff_b0_model"


def build_fallback_model():
    base = DenseNet121(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(5, activation="softmax")(x)
    model = keras.Model(inputs=base.input, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def get_input_base_dir():
    candidates = [
        "../input/aptos2019-blindness-detection",
        "/kaggle/input/aptos2019-blindness-detection",
        "../kaggle/input/aptos2019-blindness-detection",
        "/kaggle/data/aptos2019-blindness-detection",
        "/kaggle/data",
        "/kaggle/data/aptos2019-blindness-detection/aptos2019-blindness-detection",
        "/kaggle/input/aptos2019-blindness-detection/aptos2019-blindness-detection",
    ]
    for c in candidates:
        if os.path.exists(c):
            if os.path.basename(c) == "data" and os.path.exists(
                os.path.join(c, "aptos2019-blindness-detection")
            ):
                return os.path.join(c, "aptos2019-blindness-detection")
            return c
    return "/kaggle/input/aptos2019-blindness-detection"


input_dir = get_input_base_dir()
train_csv_path = os.path.join(input_dir, "train.csv")
test_csv_path = os.path.join(input_dir, "test.csv")
sample_sub_path = os.path.join(input_dir, "sample_submission.csv")
train_img_dir = os.path.join(input_dir, "train_images")
test_img_dir = os.path.join(input_dir, "test_images")

_MAP_PARALLEL = tf.data.AUTOTUNE


# SPEED FIX: Avoid re-creating the directory tf.constant per call; it is pure and constant.
def _make_full_paths_tf(images_dir, ids_tensor, _images_dir_t=None):
    if _images_dir_t is None:
        _images_dir_t = tf.constant(str(images_dir))
    # ids_tensor already strings in our usage below; keeping join semantics identical.
    return tf.strings.join([_images_dir_t, os.sep, ids_tensor, ".png"])


if os.path.exists(EFF_MODEL_PATH):
    model = keras.models.load_model(EFF_MODEL_PATH, compile=False)
    try:
        model.compile(
            optimizer=keras.optimizers.Adam(1e-4),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )
    except Exception:
        pass
else:
    train_df = pd.read_csv(train_csv_path)
    trn_df, val_df = train_test_split(
        train_df, test_size=0.15, random_state=SEED, stratify=train_df["diagnosis"]
    )

    def make_dataset(df, images_dir, training, cache=False, cache_path=None):
        ids = df["id_code"].astype(str).values
        labels_np = df["diagnosis"].values.astype("int64")

        # SPEED FIX: Build paths once using TF string tensors; avoid repeated casting.
        ids_t = tf.constant(ids, dtype=tf.string)
        images_dir_t = tf.constant(str(images_dir))
        paths_t = _make_full_paths_tf(images_dir, ids_t, _images_dir_t=images_dir_t)
        labels_t = tf.constant(labels_np, dtype=tf.int64)

        ds = tf.data.Dataset.from_tensor_slices((paths_t, labels_t))

        options = tf.data.Options()
        options.experimental_deterministic = True
        ds = ds.with_options(options)

        if training:
            ds = ds.shuffle(
                min(int(df.shape[0]), 2048), seed=SEED, reshuffle_each_iteration=True
            )

        def _map(path, label):
            img = _tf_load_ben_preprocess(path)
            label.set_shape(())
            return img, label

        ds = ds.map(_map, num_parallel_calls=_MAP_PARALLEL, deterministic=True)

        # SPEED FIX: Writing cache to disk is very slow and not needed for a single short run.
        # To preserve semantics (and still benefit on repeated executions), only use disk cache
        # if it already exists; otherwise skip caching.
        if cache:
            if cache_path and tf.io.gfile.exists(cache_path):
                ds = ds.cache(cache_path)

        ds = ds.batch(BATCH_SIZE, drop_remainder=False)
        ds = ds.prefetch(tf.data.AUTOTUNE)
        return ds

    train_ds = make_dataset(
        trn_df,
        train_img_dir,
        training=True,
        cache=True,
        cache_path="train_cache.tfdata",
    )
    val_ds = make_dataset(
        val_df, train_img_dir, training=False, cache=True, cache_path="val_cache.tfdata"
    )

    cw = class_weight.compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1, 2, 3, 4]),
        y=trn_df["diagnosis"].values,
    )
    cw = {i: float(w) for i, w in enumerate(cw)}

    model = build_fallback_model()
    callbacks = [
        ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, verbose=1, min_lr=1e-6
        ),
        ModelCheckpoint(
            "fallback_best.keras", monitor="val_loss", save_best_only=True, verbose=0
        ),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=3,  # unchanged core logic
        class_weight=cw,
        callbacks=callbacks,
        verbose=1,
    )
    if os.path.exists("fallback_best.keras"):
        model = keras.models.load_model("fallback_best.keras")

    del train_ds, val_ds, train_df, trn_df, val_df
    gc.collect()




In [ ]:
test_csv = pd.read_csv(test_csv_path)
id_code = test_csv["id_code"].astype(str).values


def make_test_dataset(ids, images_dir):
    # SPEED FIX: Avoid redundant astype/str conversions and avoid slow on-disk cache writes.
    ids_t = tf.constant(ids, dtype=tf.string)
    images_dir_t = tf.constant(str(images_dir))
    full_paths_t = _make_full_paths_tf(images_dir, ids_t, _images_dir_t=images_dir_t)

    ds = tf.data.Dataset.from_tensor_slices(full_paths_t)
    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)

    def _map(path):
        img = _tf_load_ben_preprocess(path)
        return img

    ds = ds.map(_map, num_parallel_calls=_MAP_PARALLEL, deterministic=True)

    # SPEED FIX: Disk caching test set is pure overhead for one predict pass.
    # Keep behavior identical by simply streaming; predictions are unchanged.
    # If a cache file already exists from a previous run, we can reuse it.
    cache_path = "test_cache.tfdata"
    if tf.io.gfile.exists(cache_path):
        ds = ds.cache(cache_path)

    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds


test_ds = make_test_dataset(id_code, test_img_dir)

pred = model.predict(test_ds, verbose=0)
test_prediction = np.asarray(np.argmax(pred, axis=1), dtype="int64")

del test_ds, pred
gc.collect()




In [ ]:
sub = pd.DataFrame({"id_code": id_code, "diagnosis": test_prediction.astype("int64")})
sub.to_csv("submission.csv", index=False)

unique, counts = np.unique(test_prediction, return_counts=True)
tmp = dict(zip(unique.tolist(), counts.tolist()))
print(tmp)
print("Wrote submission.csv with shape:", sub.shape)
print("Done!")